In [1]:
import pandas as pd
import re
from pathlib import Path

# Path to your combined sentiment dataset
DATA_DIR = Path("data")
INPUT_PATH = DATA_DIR / "news_sentiment_final.csv"
OUTPUT_PATH = DATA_DIR / "ai_news_corpus_for_rag.csv"

INPUT_PATH, OUTPUT_PATH


(PosixPath('data/news_sentiment_final.csv'),
 PosixPath('data/ai_news_corpus_for_rag.csv'))

In [2]:
df = pd.read_csv(INPUT_PATH)
print(df.shape)
df.head()


(3547, 6)


,source,title,description,content,clean_content,sentiment
0,Al Jazeera,What is the political agenda of artificial int...,"May 17, 2023 ... We do not believe that it cou...",Could AI single-handedly decide the course of ...,Could AI single-handedly decide the course of ...,1
1,Al Jazeera,Facebook reveals AI use to block 'terrorist co...,"Jun 16, 2017 ... Facebook says it has stepped ...",US company says technology used to block child...,US company says technology used to block child...,1
2,Al Jazeera,Could artificial intelligence lead to world pe...,"May 30, 2017 ... Machines and artificial intel...",Can one man with terminal cancer complete his ...,Can one man with terminal cancer complete his ...,4
3,Al Jazeera,Artificial intelligence without borders | US-M...,"Jun 9, 2023 ... The US-Mexico border is alread...",The US border-industrial complex has joined th...,The US border-industrial complex has joined th...,4
4,Al Jazeera,Google shows the AI evolution of its search en...,"May 11, 2023 ... Google is rolling out more ar...",Google is rolling out more artificial intellig...,Google is rolling out more artificial intellig...,5


In [3]:
# Drop exact duplicate titles, keep the first occurrence
df_dedup = df.drop_duplicates(subset=["title"]).reset_index(drop=True)

print("Original rows:", len(df))
print("After dropping duplicate titles:", len(df_dedup))


Original rows: 3547
After dropping duplicate titles: 1498


In [4]:
# Combine key text fields into one string for topic detection
text_cols = ["title", "description", "clean_content"]

for col in text_cols:
    if col not in df_dedup.columns:
        df_dedup[col] = ""

df_dedup["combined_text"] = (
    df_dedup["title"].fillna("") + " " +
    df_dedup["description"].fillna("") + " " +
    df_dedup["clean_content"].fillna("")
)


In [5]:
# Regex pattern for AI-related topics
ai_pattern = re.compile(
    r"\b("
    r"artificial intelligence|"
    r"machine learning|deep learning|neural network[s]?|"
    r"chatgpt|gpt-4|gpt4|gpt-3\.5|"
    r"large language model[s]?|llm[s]?|"
    r"generative ai|"
    r"openai|bard|gemini|copilot|"
    r"robotic[s]?|automation|autonomous system[s]?|"
    r"computer vision"
    r")\b",
    flags=re.IGNORECASE
)


In [6]:
ai_mask = df_dedup["combined_text"].str.contains(ai_pattern, na=False)

ai_df = df_dedup[ai_mask].copy().reset_index(drop=True)

print("AI-related articles:", len(ai_df))
ai_df[["source", "title"]].head(10)


/var/folders/xv/rj070sz972sf0vvfm1w4m5ph0000gn/T/ipykernel_83422/2052929011.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ai_mask = df_dedup["combined_text"].str.contains(ai_pattern, na=False)


AI-related articles: 463


,source,title
0,Al Jazeera,What is the political agenda of artificial int...
1,Al Jazeera,Facebook reveals AI use to block 'terrorist co...
2,Al Jazeera,Could artificial intelligence lead to world pe...
3,Al Jazeera,Artificial intelligence without borders | US-M...
4,Al Jazeera,Google shows the AI evolution of its search en...
5,Al Jazeera,"Paris AI summit: Why won't US, UK sign global ..."
6,Al Jazeera,Artificial intelligence must not exacerbate in...
7,Al Jazeera,UN approves its first resolution on artificial...
8,Al Jazeera,EU parliament greenlights landmark artificial ...
9,Al Jazeera,Artificial intelligence: friend or foe? | Scie...


In [7]:
# Add a stable article_id for citation and chunk metadata
ai_df["article_id"] = range(1, len(ai_df) + 1)

# Keep only the columns we need for RAG + citation
cols_order = [
    "article_id",
    "source",
    "title",
    "description",
    "content",
    "clean_content",
    "sentiment",
]

ai_df_final = ai_df[cols_order].copy()

# Save to CSV
ai_df_final.to_csv(OUTPUT_PATH, index=False)

print("Saved AI corpus for RAG to:", OUTPUT_PATH)
print("Final shape:", ai_df_final.shape)
ai_df_final.head()


Saved AI corpus for RAG to: data/ai_news_corpus_for_rag.csv
Final shape: (463, 7)


,article_id,source,title,description,content,clean_content,sentiment
0,1,Al Jazeera,What is the political agenda of artificial int...,"May 17, 2023 ... We do not believe that it cou...",Could AI single-handedly decide the course of ...,Could AI single-handedly decide the course of ...,1
1,2,Al Jazeera,Facebook reveals AI use to block 'terrorist co...,"Jun 16, 2017 ... Facebook says it has stepped ...",US company says technology used to block child...,US company says technology used to block child...,1
2,3,Al Jazeera,Could artificial intelligence lead to world pe...,"May 30, 2017 ... Machines and artificial intel...",Can one man with terminal cancer complete his ...,Can one man with terminal cancer complete his ...,4
3,4,Al Jazeera,Artificial intelligence without borders | US-M...,"Jun 9, 2023 ... The US-Mexico border is alread...",The US border-industrial complex has joined th...,The US border-industrial complex has joined th...,4
4,5,Al Jazeera,Google shows the AI evolution of its search en...,"May 11, 2023 ... Google is rolling out more ar...",Google is rolling out more artificial intellig...,Google is rolling out more artificial intellig...,5
